# M2 — Agent Layer Demo

End-of-milestone demonstration: all three agents (news sentiment, macro regime, Polymarket events).

## Prerequisites
```bash
uv run python scripts/init_db.py
uv run python scripts/ingest_prices.py
uv run python scripts/ingest_macro.py
uv run python scripts/ingest_alpha_vantage_news.py  # historical backfill
uv run python scripts/ingest_polymarket.py
```

## Sections
1. **Run NewsAgent** — news sentiment signals for one week
2. **Sentiment bar chart** — per-sector scores
3. **Run MacroAgent** — regime classification + rate outlook
4. **Macro regime timeline** — classified regime across the backtest window
5. **Run PolymarketAgent** — sector tilts driven by prediction markets
6. **Polymarket tilt chart** — sector tilts + driving events
7. **All signals table** — every signal row in SQLite

In [ ]:
import datetime
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

DB_PATH = Path('..') / 'data' / 'state.db'
engine = create_engine(f'sqlite:///{DB_PATH}')

ANALYSIS_DATE = datetime.date(2024, 6, 7)

SECTOR_NAMES = {
    'XLK': 'Technology',          'XLF': 'Financials',
    'XLV': 'Health Care',         'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',    'XLE': 'Energy',
    'XLI': 'Industrials',         'XLB': 'Materials',
    'XLRE': 'Real Estate',        'XLU': 'Utilities',
}

print(f'Analysis date : {ANALYSIS_DATE}')
print(f'DB path       : {DB_PATH.resolve()}')

## Section 1 — Run NewsAgent

Queries `news_raw` for the trailing 7 days, calls `claude-haiku-4-5-20251001` (cached on repeat),
writes 10 signal rows to `signals`.

In [ ]:
from agents.news_agent import NewsAgent

news_agent = NewsAgent()
news_input = news_agent.prepare_input(ANALYSIS_DATE, engine)
coverage = {etf: len(articles) for etf, articles in news_input['sectors'].items()}
total_articles = sum(coverage.values())

print(f'Article coverage for week {news_input["week_start"]} → {news_input["analysis_date"]}:')
for etf, n in sorted(coverage.items()):
    print(f'  {etf:5s}  {n:3d}  {"█" * min(n, 20)}')
print(f'\nTotal: {total_articles} articles')

if total_articles == 0:
    print('\n⚠  No news — run ingest_alpha_vantage_news.py first.')
    news_result = None
else:
    news_result = news_agent.run(ANALYSIS_DATE, engine)
    print(f'\n✓ NewsAgent done  conviction={news_result["conviction"]:.2f}')
    print(f'  key_themes: {news_result["key_themes"]}')

## Section 2 — Sentiment bar chart

In [ ]:
if news_result is None:
    print('⚠  No result to plot.')
else:
    sentiments = news_result['sector_sentiments']
    labels = [f'{etf} — {SECTOR_NAMES.get(etf, etf)}' for etf in sentiments]
    scores = list(sentiments.values())
    colors = ['#2ecc71' if s > 0.05 else '#e74c3c' if s < -0.05 else '#bdc3c7' for s in scores]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(labels, scores, color=colors, alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-1.05, 1.05)
    ax.set_xlabel('Sentiment  (−1 = strong bearish, +1 = strong bullish)', fontsize=10)
    ax.set_title(
        f'News Sentiment by Sector  |  week ending {ANALYSIS_DATE}  '
        f'|  conviction = {news_result["conviction"]:.2f}', fontsize=11)
    for bar, score in zip(bars, scores):
        ha = 'left' if score >= 0 else 'right'
        ax.text(score + (0.02 if score >= 0 else -0.02),
                bar.get_y() + bar.get_height() / 2,
                f'{score:+.2f}', va='center', ha=ha, fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print('Key themes:')
    for i, t in enumerate(news_result['key_themes'], 1):
        print(f'  {i}. {t}')

## Section 3 — Run MacroAgent

Pulls 30 days of FRED series, computes derived features, adds XLF+XLI news digest,
calls `claude-sonnet-4-6` with chain-of-thought reasoning.

In [ ]:
from agents.macro_agent import MacroAgent

macro_agent = MacroAgent()
macro_input = macro_agent.prepare_input(ANALYSIS_DATE, engine)

n_series = sum(len(v) for v in macro_input['series_30d'].values())
print(f'Macro data points (30d): {n_series}  |  derived features: {len(macro_input["derived_features"])}')

if macro_input['derived_features']:
    print('\nKey derived features:')
    for k, v in macro_input['derived_features'].items():
        print(f'  {k:<30s} {v}')

if n_series == 0:
    print('\n⚠  No macro data — run ingest_macro.py first.')
    macro_result = None
else:
    macro_result = macro_agent.run(ANALYSIS_DATE, engine)
    print(f'\n✓ MacroAgent done')
    print(f'  regime      : {macro_result["regime"]}')
    print(f'  rate_outlook: {macro_result["rate_outlook"]}')
    print(f'  confidence  : {macro_result["confidence"]:.2f}')
    print(f'\nRationale: {macro_result["rationale"]}')
    print(f'\nReasoning (excerpt):\n  {macro_result["reasoning"][:400]}...')

## Section 4 — Macro regime timeline

Populated after running MacroAgent across multiple dates. See loop example in the cell.

In [ ]:
regime_df = pd.read_sql(
    text("""
        SELECT s.date, s.signal_value, s.confidence,
               r.signal_value as rate_val
        FROM signals s
        LEFT JOIN signals r ON r.date = s.date AND r.agent_name = 'macro' AND r.target = 'rate_outlook'
        WHERE s.agent_name = 'macro' AND s.target = 'macro_regime'
        ORDER BY s.date
    """),
    engine, parse_dates=['date'],
)

if regime_df.empty:
    print('⚠  No regime signals yet. To populate, run:')
    print('   from agents.macro_agent import MacroAgent')
    print('   agent = MacroAgent()')
    print('   for d in pd.date_range("2024-01-05", "2024-06-07", freq="W-FRI"):')
    print('       agent.run(d.date(), engine)')
else:
    REGIME_COLORS = {1.0: '#2ecc71', 0.0: '#f39c12', -1.0: '#e74c3c'}
    REGIME_LABELS = {1.0: 'risk_on', 0.0: 'neutral', -1.0: 'risk_off'}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1]})
    for _, row in regime_df.iterrows():
        color = REGIME_COLORS.get(row['signal_value'], '#95a5a6')
        ax1.axvspan(row['date'] - pd.Timedelta(days=3), row['date'] + pd.Timedelta(days=3),
                    alpha=0.4 * row['confidence'] + 0.1, color=color, linewidth=0)
    ax1.plot(regime_df['date'], regime_df['signal_value'], 'o-', color='black', markersize=5, linewidth=1, alpha=0.6)
    ax1.set_yticks([-1, 0, 1]); ax1.set_yticklabels(['risk_off', 'neutral', 'risk_on'])
    ax1.set_title('Macro Regime Classification History', fontsize=12); ax1.grid(axis='x', alpha=0.3)
    patches = [mpatches.Patch(color=c, label=l) for v, c in REGIME_COLORS.items() for lv, l in REGIME_LABELS.items() if lv == v]
    ax1.legend(handles=patches, loc='upper right', fontsize=9)
    ax2.fill_between(regime_df['date'], regime_df['confidence'], alpha=0.5, color='steelblue')
    ax2.set_ylim(0, 1); ax2.set_ylabel('Confidence'); ax2.grid(axis='x', alpha=0.3)
    plt.tight_layout(); plt.show()

    regime_df['regime'] = regime_df['signal_value'].map(REGIME_LABELS)
    regime_df['rate'] = regime_df['rate_val'].map({1.0: '▲ rising', 0.0: '─ stable', -1.0: '▼ falling'})
    display(regime_df[['date', 'regime', 'rate', 'confidence']]
            .assign(date=regime_df['date'].dt.strftime('%Y-%m-%d'))
            .rename(columns={'date': 'Date', 'regime': 'Regime', 'rate': 'Rate Outlook', 'confidence': 'Confidence'}))

## Section 5 — Run PolymarketAgent

Reads current implied probabilities from `polymarket_raw`, fetches 30-day trend,
applies sector-impact mappings from `config/polymarket_markets.yaml`,
calls `claude-haiku-4-5-20251001` to compute aggregate sector tilts.

In [ ]:
from agents.polymarket_agent import PolymarketAgent

poly_agent = PolymarketAgent()
poly_input = poly_agent.prepare_input(ANALYSIS_DATE, engine)

markets = poly_input['markets']
n_with_data = sum(1 for m in markets if m['current_prob'] is not None)

print(f'Curated markets : {len(markets)}')
print(f'With DB data    : {n_with_data}')
print()

for m in markets:
    prob_str = f"{m['current_prob']:.2f}" if m['current_prob'] is not None else 'n/a '
    trend_str = ''
    if m['current_prob'] is not None and m['prob_30d_ago'] is not None:
        delta = m['current_prob'] - m['prob_30d_ago']
        trend_str = f'  Δ30d={delta:+.2f}'
    vol_str = f"${m['volume_usd']/1e3:.0f}k" if m['volume_usd'] else '—'
    days_str = f"{m['days_to_resolution']}d" if m['days_to_resolution'] is not None else '—'
    print(f"  [{m['confidence_rating']:6s}] p={prob_str}{trend_str}  vol={vol_str:7s}  tte={days_str:6s}  {m['question'][:60]}")

if n_with_data == 0:
    print('\n⚠  No Polymarket data — run ingest_polymarket.py first.')
    poly_result = None
else:
    poly_result = poly_agent.run(ANALYSIS_DATE, engine)
    print(f'\n✓ PolymarketAgent done  confidence={poly_result["overall_confidence"]:.2f}  horizon={poly_result["time_horizon"]}')

## Section 6 — Polymarket tilt chart

Aggregate sector tilt scores from all Polymarket events, with driving events annotated.

In [ ]:
if poly_result is None:
    print('⚠  No Polymarket result to plot.')
else:
    impacts = poly_result['sector_impacts']
    driving = poly_result['driving_events']

    etfs   = list(impacts.keys())
    tilts  = list(impacts.values())
    labels = [f'{etf} — {SECTOR_NAMES.get(etf, etf)}' for etf in etfs]
    colors = ['#2ecc71' if t > 0.02 else '#e74c3c' if t < -0.02 else '#bdc3c7' for t in tilts]

    fig, ax = plt.subplots(figsize=(11, 7))
    bars = ax.barh(labels, tilts, color=colors, alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-0.75, 0.75)
    ax.set_xlabel('Sector tilt  (−1 = strong bearish, +1 = strong bullish)', fontsize=10)
    ax.set_title(
        f'Polymarket Events — Sector Tilts  |  {ANALYSIS_DATE}  '
        f'|  horizon={poly_result["time_horizon"]}  '
        f'|  confidence={poly_result["overall_confidence"]:.2f}',
        fontsize=11)

    for bar, etf, tilt in zip(bars, etfs, tilts):
        # Score label
        ha = 'left' if tilt >= 0 else 'right'
        ax.text(tilt + (0.01 if tilt >= 0 else -0.01),
                bar.get_y() + bar.get_height() / 2,
                f'{tilt:+.2f}', va='center', ha=ha, fontsize=9)
        # Driving events annotation (first one, truncated)
        events = driving.get(etf, [])
        if events:
            short = events[0][:40] + ('…' if len(events[0]) > 40 else '')
            x_pos = tilt + (0.08 if tilt >= 0 else -0.08)
            ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                    short, va='center', ha=ha, fontsize=7, alpha=0.65,
                    style='italic')

    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Driving events detail
    print('\nDriving events per sector:')
    for etf, events in sorted(driving.items()):
        tilt = impacts.get(etf, 0)
        print(f'  {etf:5s} ({tilt:+.2f}):')
        for e in events:
            print(f'    • {e}')

## Section 7 — All signals in DB

## Section 8 — Full pipeline run

`run_agent_pipeline` runs all three agents then the aggregator in one call.
Failed agents are replaced with neutral (zero) stubs — the pipeline never crashes
on a single agent error.

In [ ]:
from agents.pipeline import run_agent_pipeline

pipeline_result = run_agent_pipeline(ANALYSIS_DATE, engine)

# ── Agent status ──────────────────────────────────────────────────────────────
print(f'Pipeline date : {pipeline_result["date"]}')
print(f'Total cost    : ${pipeline_result["total_cost_usd"]:.5f}')
print(f'Total latency : {pipeline_result["total_latency_ms"]:.0f} ms')
print()
print('Agent status:')
for name, info in pipeline_result['signals_by_agent'].items():
    status = info['status']
    latency = info.get('latency_ms', 0)
    if status == 'ok':
        print(f'  {name:<12s} ✓  {latency:.0f} ms')
    else:
        print(f'  {name:<12s} ✗  {latency:.0f} ms  ERROR: {info["error"][:80]}')

# ── Views (Q vector) ──────────────────────────────────────────────────────────
print()
print('Black-Litterman views (Q = weekly expected excess return):')
sectors = list(SECTOR_NAMES.keys())
q_values = pipeline_result['views']['q']
omega_diag = pipeline_result['views']['omega_diag']

for etf, q, omega in zip(sectors, q_values, omega_diag):
    bar_len = int(abs(q) / (0.05 / 52) * 20)  # scale: unit signal = 20 chars
    bar = ('█' * bar_len) if q >= 0 else ('░' * bar_len)
    direction = '▲' if q > 1e-6 else '▼' if q < -1e-6 else '─'
    print(f'  {etf:5s}  Q={q:+.5f}  Ω={omega:.4f}  {direction} {bar}')

# ── Views chart ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if q > 1e-6 else '#e74c3c' if q < -1e-6 else '#bdc3c7' for q in q_values]
labels = [f'{etf} — {SECTOR_NAMES[etf]}' for etf in sectors]
bars = ax.barh(labels, q_values, color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)

# Mark ±1 signal equivalent as reference lines
max_weekly = 0.05 / 52
ax.axvline(max_weekly, color='grey', linewidth=0.6, linestyle='--', alpha=0.6, label='±1σ signal')
ax.axvline(-max_weekly, color='grey', linewidth=0.6, linestyle='--', alpha=0.6)

for bar, q in zip(bars, q_values):
    if abs(q) > 1e-6:
        ha = 'left' if q >= 0 else 'right'
        ax.text(q + (1e-5 if q >= 0 else -1e-5), bar.get_y() + bar.get_height() / 2,
                f'{q:+.5f}', va='center', ha=ha, fontsize=8)

ax.set_xlabel('Weekly expected excess return', fontsize=10)
ax.set_title(
    f'Black-Litterman Views (Q)  |  {pipeline_result["date"]}  '
    f'|  pipeline cost=${pipeline_result["total_cost_usd"]:.4f}',
    fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
all_sig_df = pd.read_sql(
    text("""
        SELECT s.date, s.agent_name, s.target, s.signal_value, s.confidence,
               a.model_string, a.cached, a.cost_usd, a.latency_ms
        FROM signals s
        LEFT JOIN agent_calls a ON s.raw_call_id = a.call_id
        ORDER BY s.date DESC, s.agent_name, s.target
    """),
    engine,
)

if all_sig_df.empty:
    print('⚠  No signals in DB yet.')
else:
    all_sig_df['signal_value'] = all_sig_df['signal_value'].round(3)
    all_sig_df['cost_usd'] = all_sig_df['cost_usd'].map(lambda x: f'${x:.5f}' if pd.notna(x) else '—')
    all_sig_df['latency_ms'] = all_sig_df['latency_ms'].map(lambda x: f'{x:.0f}ms' if pd.notna(x) else '—')
    all_sig_df['cached'] = all_sig_df['cached'].map(lambda x: '✓' if x else '✗')
    display(all_sig_df)
    print(f'\nTotal signal rows: {len(all_sig_df)}')

    # Cost summary
    cost_df = pd.read_sql(
        text("SELECT agent_name, SUM(cost_usd) as total_cost, COUNT(*) as calls FROM agent_calls GROUP BY agent_name"),
        engine,
    )
    if not cost_df.empty:
        print('\nLLM cost summary:')
        display(cost_df)